# Module 2 - Index the OKF store: Qdrant (dense + BM25) and Neo4j (graph)

**EN.** Session 1 turned the raw documents into OKF files (`data/okf_store/`). This notebook **indexes** them in two databases, the way Track A (GraphRAG + hybrid search) will use them:

```
data/okf_store/*.md ──► chunks ──┬──► Qdrant: 2 vectors per chunk (dense = meaning, BM25 = exact words)
                                 └──► Neo4j : entities + relationships (the graph)
```

Then it tests the three kinds of question below, **with the evidence printed**: which method finds the right chunk, and which one fails.

**PT.** Na sessão 1 transformámos os documentos em ficheiros OKF (`data/okf_store/`). Este notebook **indexa-os** em duas bases de dados, tal como o Track A (GraphRAG + hybrid search) as vai usar: cada chunk vai para o Qdrant com dois vetores (dense = significado, BM25 = palavras exatas) e as entidades e relações vão para o Neo4j (o grafo). Depois testamos os três tipos de pergunta da tabela, **com a prova impressa**: que método encontra o chunk certo e qual falha.

| Question type / Tipo de pergunta | Dense | BM25 | Graph | Shown in / Onde se vê |
|---|---|---|---|---|
| Paraphrase: "my fan stopped spinning" | good | weak | - | section 5b |
| Exact token: `FM4`, `0x4A7B_VRM_OVERVOLT` | unreliable | excellent | - | sections 5a, 5c |
| Relations: "which thermal codes light a solid red LED?" | weak | weak | excellent | section 4 |

Run it cell by cell (Shift+Enter). If the databases are already full, section 3 simply rebuilds them (it is idempotent).
*Corre célula a célula (Shift+Enter). Se as bases de dados já estiverem cheias, a secção 3 reconstrói-as (é idempotente).*

In [ ]:
import os, sys, json, asyncio, time, logging
from pathlib import Path
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)                                   # every path below is relative to the project root
os.environ.setdefault("HF_HOME", str(ROOT / "models" / "huggingface"))
logging.getLogger("neo4j").setLevel(logging.ERROR)
MODEL = os.getenv("FORMATION_MODEL", "formation-fast")   # use "formation-local" if you run LM Studio / Ollama

import warnings; warnings.filterwarnings("ignore")
logging.getLogger("httpx").setLevel(logging.WARNING)          # hide one log line per Qdrant request
from backend.graph_rag.neo4j_client import Neo4jGraphClient
from backend.graph_rag.qdrant_hybrid import QdrantHybridStore
from backend.graph_rag.entity_extractor import EntityExtractor
from backend.graph_rag.chunker import chunk_okf_document, split_frontmatter
from backend.graph_rag.indexer import index_okf_store, build_chunks, okf_files
neo4j, qdrant = Neo4jGraphClient(), QdrantHybridStore()
print("Neo4j connected :", neo4j.driver is not None)
print("Qdrant ready    :", qdrant.ready, "(dense + BM25 models loaded)")

**Result / Resultado.** Two `True`: the notebook talks to both databases running in Docker, and the two models that turn text into vectors are loaded locally (`bge-small-en-v1.5` for dense, a BM25 tokenizer for sparse).
*Dois `True`: o notebook fala com as duas bases de dados no Docker, e os dois modelos que transformam texto em vetores estão carregados localmente.*

## 1. Chunking that respects structure / Chunking que respeita a estrutura

**EN.** A model reads only a few chunks (we give it 5), so *how* we cut the documents decides what it can see. Tables are split **one row per chunk**, prefixed with the column names; prose is packed up to ~900 characters; each chunk keeps its heading path and the figures it mentions.

**PT.** O modelo só lê alguns chunks (damos-lhe 5), por isso *a forma como cortamos* os documentos decide o que ele consegue ver. As tabelas são cortadas **uma linha por chunk**, com o nome das colunas à frente; o texto corrido é agrupado até ~900 caracteres; cada chunk guarda o caminho de títulos e as figuras que menciona.

In [ ]:
chunks = build_chunks("hardware_manuals")
print(len(chunks), "chunks in the hardware corpus;", len(build_chunks("syone_corp")), "in the Syone corpus")
row = next(c for c in chunks if c["kind"] == "table_row" and "0x4A7B_VRM_OVERVOLT" in c["text"])
print("\nA table-row chunk:\n", row["text"])
fig = next(c for c in chunks if c["images"])
print("\nA chunk that carries figures:", fig["section"], "->", [i["figure_id"] for i in fig["images"]])

**Result / Resultado.**
* **EN.** 714 chunks for 8 manuals, 1,373 for 79 web pages. The table-row chunk reads `Quantity: Overvoltage protection cutoff | Threshold or value: 54.2 V | Related fault code: 0x4A7B_VRM_OVERVOLT`: the number **54.2** keeps its meaning because the column names travel with it. A splitter that cuts every 500 characters would separate the number from its header. The last line shows a chunk that carries its figures: when it is retrieved, the figure comes with it.
* **PT.** 714 chunks para 8 manuais, 1.373 para 79 páginas web. O chunk da linha de tabela diz `Quantity ... 54.2 V ... 0x4A7B_VRM_OVERVOLT`: o número **54.2** mantém o significado porque os nomes das colunas vão com ele. Um splitter que corta a cada 500 caracteres separava o número do cabeçalho. A última linha mostra um chunk que leva as suas figuras: quando é encontrado, a figura vem junto.

## 2. Entities and relationships for the graph / Entidades e relações para o grafo

**EN.** From the same OKF file we extract *things* (fault codes, LED states, parts, subsystems) and the *relations* between them. Rule-based and **data-driven**: the facts are read from the document's tables, not hard-coded. In production an LLM extractor is the usual next step; the graph model stays the same.

**PT.** Do mesmo ficheiro OKF extraímos *coisas* (códigos de erro, estados de LED, peças, subsistemas) e as *relações* entre elas. É por regras e **guiado pelos dados**: os factos são lidos das tabelas do documento, não estão escritos no código. Em produção, o passo seguinte habitual é um extrator com LLM; o modelo do grafo mantém-se.

In [ ]:
ex = EntityExtractor().extract_from_okf_file(ROOT / "data/okf_store/hardware_manuals/aor3_06_fault_code_reference.md")
print(len(ex["nodes"]), "nodes,", len(ex["relationships"]), "relationships from one document")
code_node = next(n for n in ex["nodes"] if n["id"] == "code:0x4A7B")
print(json.dumps(code_node, indent=1))
print([ (r["type"], r["target"]) for r in ex["relationships"] if r["source"] == "code:0x4A7B"])

**Result / Resultado.**
* **EN.** 68 nodes and 271 relationships from **one** manual. The node `0x4A7B` carries its subsystem, severity, LED, root cause and remediation as properties, and the last line lists its arrows: `IN_SUBSYSTEM` Power VRM, `INDICATED_BY` Solid red PWR, ... Each arrow is one cell of the fault table.
* **PT.** 68 nós e 271 relações a partir de **um** manual. O nó `0x4A7B` guarda o subsistema, a severidade, o LED, a causa e a remediação como propriedades, e a última linha lista as setas: `IN_SUBSYSTEM` Power VRM, `INDICATED_BY` Solid red PWR, ... Cada seta é uma célula da tabela de códigos.

## 3. Index everything / Indexar tudo

**EN.** For every chunk: compute the dense vector and the BM25 vector, store both in Qdrant; write the entities and relations to Neo4j. Idempotent: running it twice gives the same result. **This cell is exactly `python -m backend.graph_rag.indexer`** in a terminal (about 40 s).

**PT.** Para cada chunk: calcula o vetor dense e o vetor BM25 e guarda os dois no Qdrant; escreve as entidades e relações no Neo4j. É idempotente: correr duas vezes dá o mesmo resultado. **Esta célula é exatamente `python -m backend.graph_rag.indexer`** num terminal (cerca de 40 s).

In [ ]:
t0 = time.time()
report = index_okf_store(neo4j, qdrant)
print(json.dumps(report, indent=1)); print(f"{time.time()-t0:.0f}s")

**Result / Resultado.**
* **EN.** Hardware: 8 documents, 714 chunks, 90 nodes, 289 relationships. Syone: 79 documents, 1,373 chunks, 137 nodes, 796 relationships. In total 2,087 points in Qdrant and 227 nodes / 1,085 relationships in Neo4j: the numbers in the console header.
* **PT.** Hardware: 8 documentos, 714 chunks, 90 nós, 289 relações. Syone: 79 documentos, 1.373 chunks, 137 nós, 796 relações. No total, 2.087 pontos no Qdrant e 227 nós / 1.085 relações no Neo4j: os números do cabeçalho da consola.

## 4. The graph answers relation questions / O grafo responde a perguntas de relações

Row 3 of the table: *"which thermal codes light a solid red LED?"* (also open http://localhost:17474, login `neo4j` / `syone2026`).

**EN.** The answer needs **two columns of 30 rows at once** (subsystem = Thermal **and** LED = solid red). A vector search returns chunks that *look like* the question; a Cypher query returns exactly the rows that *satisfy* it. Cypher in one line: `MATCH` a drawn shape `(node)-[:RELATION]->(node)`, `WHERE` a filter, `RETURN` what to show.

**PT.** A resposta precisa de **duas colunas de 30 linhas ao mesmo tempo** (subsistema = Thermal **e** LED = vermelho fixo). Uma pesquisa vetorial devolve chunks que *se parecem* com a pergunta; uma query Cypher devolve exatamente as linhas que a *satisfazem*. Cypher numa linha: `MATCH` uma forma desenhada `(nó)-[:RELAÇÃO]->(nó)`, `WHERE` um filtro, `RETURN` o que mostrar.

In [ ]:
import pandas as pd
print("Node counts:"); display(pd.DataFrame(neo4j.stats()["nodes"]).T)
q = """MATCH (c:ErrorCode)-[:IN_SUBSYSTEM]->(:Subsystem {name:'Thermal'}), (c)-[:INDICATED_BY]->(l:LEDState)
       WHERE l.name CONTAINS 'Solid red' RETURN c.name AS code, c.severity AS severity, c.remediation AS remediation"""
display(pd.DataFrame(neo4j.run_cypher(q)))

**Result / Resultado.**
* **EN.** The node counts: Document 87, ErrorCode 30, Technology 25, Part 18, ... Then exactly **two** codes: `0x8F11_TEMP_CRIT` and `0x8F02_FAN_ALL_FAIL`, both Critical, with their remediation. No ranking, no "top 5": the complete and exact answer.
* **PT.** As contagens de nós: Document 87, ErrorCode 30, Technology 25, Part 18, ... Depois exatamente **dois** códigos: `0x8F11_TEMP_CRIT` e `0x8F02_FAN_ALL_FAIL`, ambos Critical, com a remediação. Sem ranking, sem "top 5": a resposta completa e exata.

The same works on the website / *O mesmo funciona no site*: Syone's partners and their tier.

In [ ]:
q = """MATCH (o:Organization)-[:PARTNERS_WITH]->(p:Partner)<-[:DESCRIBES]-(d:Document) RETURN p.name AS partner, p.tier AS tier, d.doc_id AS page ORDER BY partner"""
display(pd.DataFrame(neo4j.run_cypher(q)))

**Result / Resultado.** Six partners (Elastic, GitLab, Microsoft, Red Hat, Syntevo, Zabbix), each with its tier and the page it came from. *Seis parceiros, cada um com o nível e a página de onde veio.*

## 5. Dense vs BM25 vs hybrid, with the ranks visible / Dense vs BM25 vs hybrid, com os rankings à vista

**EN.** Each method returns its 20 best chunks. **Reciprocal Rank Fusion (RRF)** gives each chunk `1/(60 + rank)` from each list and adds them up; the 5 best go to the model. Every line below shows:

`RRF <fused score> | dense #<rank> | bm25 #<rank> | <document> | <start of the chunk>`

`None` = that method did not put the chunk in its top 20. Example: dense #2 and BM25 #12 give 1/62 + 1/72 = 0.0161 + 0.0139 = **0.0300**. A chunk found by only one method scores at most 1/61 = 0.0164, so **agreement between the two methods wins**. This is *debuggable retrieval*: you can always see why a chunk won.

**PT.** Cada método devolve os seus 20 melhores chunks. O **Reciprocal Rank Fusion (RRF)** dá a cada chunk `1/(60 + rank)` de cada lista e soma; os 5 melhores vão para o modelo. Cada linha abaixo mostra o score fundido, o rank no dense, o rank no BM25, o documento e o início do chunk. `None` = esse método não pôs o chunk no seu top 20. Exemplo: dense #2 e BM25 #12 dão 1/62 + 1/72 = **0.0300**. Um chunk encontrado só por um método tem no máximo 1/61 = 0.0164, por isso **o acordo entre os dois métodos ganha**. É *retrieval que se consegue depurar*: vê-se sempre porque é que um chunk ganhou.

### 5a. An exact fault code / Um código de erro exato

In [ ]:
def show(query, corpus="hardware_manuals", k=4):
    print(f"\nQUERY: {query}")
    for h in qdrant.hybrid_search(query, corpus=corpus, top_k=k):
        print(f"  RRF {h['hybrid_score']:.4f} | dense #{str(h['dense_rank']):>4} | bm25 #{str(h['bm25_rank']):>4} | {h['title'][:34]:34s} | {h['text'].split(chr(10),1)[1][:70]}")

show("0x4A7B_VRM_OVERVOLT")   # the full code
show("0x4A7B")                # only the prefix: works because we also index the split form of every code

**Result / Resultado.**
* **EN.** For the full code, the **definition row** (`Code: 0x4A7B_VRM_OVERVOLT | Subsystem: Power VRM | Severity: Critical ...`) is dense #2 but only **BM25 #12**: the code appears in about 15 chunks across the manuals, and BM25 prefers short chunks that repeat it. First place goes to a paragraph about the SEL export that also names the code (dense #4, BM25 #2). Fusion brings the definition row to **2nd**, inside the 5 chunks the model reads. Neither method alone is "the right one"; the fusion is robust. With the full question from the console ("What is the root cause ... for 0x4A7B_VRM_OVERVOLT?") the row is #1 for both methods. With only the prefix `0x4A7B`, BM25 #1 is the table that lists every Power VRM code: it still works because each code is also indexed in its split form.
* **PT.** Para o código completo, a **linha de definição** é dense #2 mas só **BM25 #12**: o código aparece em cerca de 15 chunks dos manuais, e o BM25 prefere chunks curtos que o repetem. O primeiro lugar vai para um parágrafo sobre o export do SEL que também cita o código (dense #4, BM25 #2). A fusão põe a linha de definição em **2.º**, dentro dos 5 chunks que o modelo lê. Nenhum método sozinho é "o certo"; a fusão é robusta. Com a pergunta completa da consola, a linha fica em #1 nos dois métodos. Só com o prefixo `0x4A7B`, o BM25 #1 é a tabela com todos os códigos Power VRM: funciona porque cada código também é indexado na forma partida.

### 5b. A paraphrase: no shared words / Uma paráfrase: nenhuma palavra em comum

Row 1 of the table. The manuals say "fan" and "tachometer"; we ask about a "cooling unit" that stopped "rotating". *Os manuais dizem "fan" e "tachometer"; nós perguntamos por uma "cooling unit" que deixou de "rotating".*

In [ ]:
show("the cooling unit stopped rotating")

**Result / Resultado.**
* **EN.** Dense #1 (switch fan trays) and dense #2 (the procedure for `0x8F01_FAN_TACH_FAIL`, a fan that stopped) both have **BM25 `None`**: no word in common, so BM25 cannot see them. Only the *meaning* carries them. BM25's own #1 is a table row about the "Fan wall" that matched a word but not the problem. The fused #1 is the first step of the cooling procedure ("Read the SEL"), ranked by *both* methods (dense #6, BM25 #2): agreement wins. **Paraphrase → dense.**
* **PT.** O dense #1 (as gavetas de ventoinhas do switch) e o dense #2 (o procedimento para `0x8F01_FAN_TACH_FAIL`, uma ventoinha parada) têm os dois **BM25 `None`**: nenhuma palavra em comum, por isso o BM25 não os vê. Só o *significado* os encontra. O #1 do próprio BM25 é uma linha sobre o "Fan wall" que acertou numa palavra, mas não no problema. O #1 da fusão é o primeiro passo do procedimento de arrefecimento ("Read the SEL"), escolhido pelos *dois* métodos (dense #6, BM25 #2): o acordo ganha. **Paráfrase → dense.**

### 5c. A short exact identifier: where dense fails / Um identificador curto e exato: onde o dense falha

Row 2 of the table. `FM4` is one fan module (the manual calls them FM1 to FM6). Here we print the top 3 of **each method on its own**, before fusion, and mark with ✔ the chunks that really contain `FM4`.
*`FM4` é um módulo de ventoinha (o manual chama-lhes FM1 a FM6). Imprimimos o top 3 de **cada método sozinho**, antes da fusão, e marcamos com ✔ os chunks que contêm mesmo `FM4`.*

In [ ]:
def compare(query, must_contain, corpus="hardware_manuals", n=3):
    hits = qdrant.hybrid_search(query, corpus=corpus, top_k=40)
    print(f"QUERY: {query}")
    for method in ("dense", "bm25"):
        ranked = sorted((h for h in hits if h[f"{method}_rank"]), key=lambda h: h[f"{method}_rank"])[:n]
        print(f"\n  {method} alone, top {n}:")
        for h in ranked:
            mark = "✔" if must_contain in h["text"] else "✘"
            print(f"    {mark} #{h[f'{method}_rank']} | {h['text'].split(chr(10),1)[1][:80]}")

compare("FM4", must_contain="FM4")

**Result / Resultado.**
* **EN.** Dense alone: **no** chunk in its top 3 contains `FM4`. It returns figure *F4*, the line "FM1 to FM6" and an LED in *position 4*: to an embedding, `FM4`, `F4` and `4` *look alike*, and a 3-character code carries almost no meaning. BM25 alone: all three contain `FM4` (the rows of `0x8F01_FAN_TACH_FAIL`, the fault for that fan). **Exact token → BM25.** This is why we keep both: part numbers, error codes, VLAN names and product SKUs are everywhere in enterprise documents.
* **PT.** Dense sozinho: **nenhum** chunk do top 3 contém `FM4`. Devolve a figura *F4*, a linha "FM1 to FM6" e um LED na *posição 4*: para um embedding, `FM4`, `F4` e `4` *parecem-se*, e um código de 3 caracteres quase não tem significado. BM25 sozinho: os três contêm `FM4` (as linhas de `0x8F01_FAN_TACH_FAIL`, o erro dessa ventoinha). **Token exato → BM25.** É por isso que mantemos os dois: números de peça, códigos de erro, nomes de VLAN e SKUs estão por todo o lado nos documentos de uma empresa.

### 5d. Generic words add noise / Palavras genéricas fazem ruído

In [ ]:
show("error 0x8F01")

**Result / Resultado.**
* **EN.** The top 4 are about **ECC memory errors**, not the fan code `0x8F01`. The word "error" is everywhere, and to the embedding "error 0x8F01" looks like "error 0x10E4". **BM25 is not magic**: one generic word drags both methods off. Remedies: a stop-word list, query rewriting, or the full question (the console asks "What does 0x8F01_FAN_TACH_FAIL mean?").
* **PT.** O top 4 é sobre **erros de memória ECC**, não sobre o código da ventoinha `0x8F01`. A palavra "error" está em todo o lado, e para o embedding "error 0x8F01" parece "error 0x10E4". **O BM25 não é magia**: uma palavra genérica desvia os dois métodos. Remédios: uma lista de stop-words, reescrever a pergunta, ou a pergunta completa.

### 5e. Near-duplicate codes: the hard case / Códigos quase iguais: o caso difícil

`0x4A7B_VRM_OVERVOLT` (above 54.2 V, critical) vs `0x4A7C_VRM_UNDERVOLT` (below 40.5 V, major): one character apart, opposite meaning.
*Um carácter de diferença, significado oposto.*

In [ ]:
show("0x4A7C_VRM_UNDERVOLT", k=3)
show("Which fault means the busbar dropped below 40.5 V?", k=3)

**Result / Resultado.**
* **EN.** For `0x4A7C`, #1 is the paragraph that *contrasts* the two codes (BM25 #1), then the `0x4A7C` definition row: the right code, not its neighbour. Asked by meaning ("below 40.5 V"), #1 is the *Undervoltage cutoff 40.5 V* row (dense #1, BM25 #2) and then the `0x4A7C` rows: both methods agree, so fusion is confident.
* **PT.** Para `0x4A7C`, o #1 é o parágrafo que *compara* os dois códigos (BM25 #1), depois a linha de definição do `0x4A7C`: o código certo, não o vizinho. Perguntando pelo significado ("below 40.5 V"), o #1 é a linha *Undervoltage cutoff 40.5 V* (dense #1, BM25 #2) e depois as linhas do `0x4A7C`: os dois métodos concordam, por isso a fusão tem confiança.

## Recap / Resumo

| Question / Pergunta | What we saw / O que vimos |
|---|---|
| Paraphrase (5b) | dense finds the fan chunks; BM25 `None` |
| Exact token (5c) | BM25 3/3 with `FM4`; dense 0/3 |
| Exact code (5a) | BM25 alone puts the definition 12th; fusion brings it to 2nd |
| Relations (4) | Cypher returns exactly the 2 codes; no ranking |
| Noise (5d) | a generic word drags both methods off |

**EN.** No single method wins everywhere: hybrid search fuses dense and BM25, and the graph answers what no ranking can. This is Track A. Next, in the console: the same questions to Track A and to Track B (an agent that reads the OKF files directly, no index).

**PT.** Nenhum método ganha em todo o lado: a hybrid search junta dense e BM25, e o grafo responde ao que nenhum ranking consegue. Isto é o Track A. A seguir, na consola: as mesmas perguntas ao Track A e ao Track B (um agente que lê os ficheiros OKF diretamente, sem índice).

### Exercises / Exercícios
1. Query `show("0x4A7B_VRM_OVERVOLT")` after replacing the BM25 sparse vector with a zero vector (hint: comment out the `bm25` prefetch in `qdrant_hybrid.py`). Which queries break?
2. Try `compare("MGMT_VLAN_102", "MGMT_VLAN_102")` and `compare("SEL-0101", "SEL-0101")`: does dense fail on every identifier, or only on short ones?
3. Write a Cypher query that returns every code raised by the `AS-100G switch` component and the LED each one shows.
4. Add a new `Event` alias to `backend/ingestion/knowledge.py`, re-index, and find the new nodes in the Neo4j browser.